# Lung-masked different-patient registration — seminar notebook

This streamlined notebook fine-tunes the completed 80k model and produces reproducible before/after registration figures for the seminar.


In [ ]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

In [ ]:
# 実行デバイス
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [ ]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

## Fine-tuning

Run this cell only when training or reproducing the fine-tuning. It saves checkpoints every 1,000 iterations.


In [ ]:
# Fine-tuning: different patients from TrainData_NoBed, preferring lung-masked data.
# This cell assumes that cells defining vxm, the wavelet pipeline, nb_features,
# analysis, synthesis_filters, MSE_Loss, and device have already been run.
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

DATA_DIR = Path('Data')
RAW_DATA_PATH = DATA_DIR / 'TrainData_NoBed.npz'
PRETRAINED_MODEL_PATH = Path('model_analysis_pipeline_pretrain.pth')

# Prefer a pre-masked volume archive if one is supplied. If none is found, this
# code looks for a lung-mask key in TrainData_NoBed.npz and masks the volumes.
MASKED_ARCHIVE_CANDIDATES = [
    DATA_DIR / 'TrainData_NoBed_LungMasked.npz',
    DATA_DIR / 'TrainData_NoBed_lung_masked.npz',
    DATA_DIR / 'TrainData_NoBed_Masked.npz',
    DATA_DIR / 'TrainData_NoBed_masked.npz',
]
VOLUME_KEYS = ('Train_lung_masked', 'train_lung_masked', 'TrainMasked', 'masked_train', 'Train')
MASK_KEYS = ('Train_lung_mask', 'train_lung_mask', 'LungMask', 'lung_mask', 'TrainMask', 'train_mask', 'Mask', 'mask')

if not PRETRAINED_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'80k pretrained checkpoint was not found: {PRETRAINED_MODEL_PATH.resolve()}'
    )


def to_n_dhw(array, label):
    """Convert supported volume layouts to (N, 128, 256, 256)."""
    array = np.asarray(array, dtype=np.float32)
    if array.ndim != 4:
        raise ValueError(f'{label} must be 4-D; got {array.shape}')
    if array.shape[1:] == (128, 256, 256):
        return array
    if array.shape[:3] == (128, 256, 256):
        return np.transpose(array, (3, 0, 1, 2))
    if array.shape[1:] == (256, 256, 128):
        return np.transpose(array, (0, 3, 1, 2))
    if array.shape[:3] == (256, 256, 128):
        return np.transpose(array, (3, 2, 0, 1))
    raise ValueError(
        f'{label} has unsupported shape {array.shape}; expected an N/D/H/W permutation '
        'compatible with (N, 128, 256, 256).'
    )


def first_available_key(archive, candidates):
    return next((key for key in candidates if key in archive.files), None)


def load_lung_preferred_training_data():
    masked_path = next((path for path in MASKED_ARCHIVE_CANDIDATES if path.exists()), None)
    archive_path = masked_path or RAW_DATA_PATH
    if not archive_path.exists():
        raise FileNotFoundError(f'Training archive was not found: {archive_path.resolve()}')

    with np.load(archive_path, allow_pickle=False) as archive:
        volume_key = first_available_key(archive, VOLUME_KEYS)
        if volume_key is None:
            raise KeyError(f'{archive_path} has no volume key. Available keys: {archive.files}')
        volumes = to_n_dhw(archive[volume_key], f'{archive_path}:{volume_key}')

        mask_key = first_available_key(archive, MASK_KEYS)
        masks = None if mask_key is None else to_n_dhw(archive[mask_key], f'{archive_path}:{mask_key}')

    if masks is not None:
        if masks.shape != volumes.shape:
            raise ValueError(f'Volume/mask shape mismatch: {volumes.shape} vs {masks.shape}')
        masks = (masks > 0).astype(np.float32)
        volumes = volumes * masks
        print(f'Using lung mask key: {mask_key} from {archive_path}')
    elif masked_path is not None:
        print(f'Using lung-masked volume archive: {archive_path}')
    else:
        print('WARNING: no lung mask or masked-volume archive was found; using unmasked volumes.')
        print('Checked masked archives:', [str(path) for path in MASKED_ARCHIVE_CANDIDATES])

    print('Fine-tuning volume shape:', volumes.shape)
    return volumes, masks, archive_path


def different_patient_generator(volumes, masks=None, batch_size=2, seed=42):
    """Yield moving/fixed batches whose patient indices are always different."""
    if len(volumes) < 2:
        raise ValueError('At least two patients are required for different-patient fine-tuning.')
    rng = np.random.default_rng(seed)
    while True:
        moving_indices = rng.integers(0, len(volumes), size=batch_size)
        fixed_indices = rng.integers(0, len(volumes), size=batch_size)
        while np.any(moving_indices == fixed_indices):
            repeated = moving_indices == fixed_indices
            fixed_indices[repeated] = rng.integers(0, len(volumes), size=repeated.sum())

        moving = torch.from_numpy(volumes[moving_indices]).unsqueeze(1)
        fixed = torch.from_numpy(volumes[fixed_indices]).unsqueeze(1)
        if masks is None:
            overlap = None
        else:
            moving_mask = torch.from_numpy(masks[moving_indices]).unsqueeze(1)
            fixed_mask = torch.from_numpy(masks[fixed_indices]).unsqueeze(1)
            overlap = moving_mask * fixed_mask
        yield moving, fixed, overlap, moving_indices, fixed_indices


def masked_mse(target, prediction, mask=None, eps=1e-6):
    squared_error = (target - prediction).square()
    if mask is None:
        return squared_error.mean()
    return (squared_error * mask).sum() / mask.sum().clamp_min(eps)


def smoothness_loss(flow):
    dz = (flow[:, :, 1:, :, :] - flow[:, :, :-1, :, :]).square().mean()
    dy = (flow[:, :, :, 1:, :] - flow[:, :, :, :-1, :]).square().mean()
    dx = (flow[:, :, :, :, 1:] - flow[:, :, :, :, :-1]).square().mean()
    return (dx + dy + dz) / 3.0


def reconstruct_warped(moving_images, fixed_images):
    moving_analysis = analysis_filter_3d(moving_images, analysis)
    fixed_analysis = analysis_filter_3d(fixed_images, analysis)
    moving_w = down_sampling_3d(moving_analysis).to(device)
    fixed_w = down_sampling_3d(fixed_analysis).to(device)
    flow = model3D(moving_w, fixed_w)
    warped_bands = [
        transformer(moving_w[:, band:band + 1], flow)
        for band in range(moving_w.shape[1])
    ]
    warped_wavelets = torch.cat(warped_bands, dim=1)
    warped_up = up_sampling_3d(warped_wavelets)
    warped_image, filtered_bands = synthesis_filter_3d(warped_up, synthesis_filters)
    return warped_image.to(device), flow, moving_w, warped_up, filtered_bands


volumes, lung_masks, archive_path = load_lung_preferred_training_data()
finetune_generator = different_patient_generator(volumes, masks=lung_masks, batch_size=2)

model3D = vxm.networks.VxmDense_128_256_256(
    (128, 256, 256), nb_features, int_steps=0
).to(device)
try:
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
except TypeError:  # PyTorch versions without weights_only
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
model3D.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model3D.train()
print(f'Loaded 80k pretrained model: {PRETRAINED_MODEL_PATH.resolve()}')

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-6)

finetune_epochs = 30000
smoothness_weight = 0.1
evaluation_every = 100
checkpoint_every = 1000
checkpoint_dir = Path('finetune_checkpoints_different_patients')
checkpoint_dir.mkdir(exist_ok=True)

losses, image_losses, smooth_losses = [], [], []

for epoch in tqdm(range(1, finetune_epochs + 1), desc='Fine-tuning'):
    moving_cpu, fixed_cpu, overlap_cpu, moving_ids, fixed_ids = next(finetune_generator)
    moving_images = moving_cpu.to(device=device, dtype=torch.float32)
    fixed_images = fixed_cpu.to(device=device, dtype=torch.float32)
    overlap_mask = None if overlap_cpu is None else overlap_cpu.to(device=device, dtype=torch.float32)

    optimizer.zero_grad(set_to_none=True)
    transformed_image, Vec, moving_w, moving_warped_up, filtered_bands = reconstruct_warped(
        moving_images, fixed_images
    )
    loss_image = masked_mse(fixed_images, transformed_image, overlap_mask)
    loss_smooth = smoothness_loss(Vec)
    loss = loss_image + smoothness_weight * loss_smooth
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model3D.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(loss.detach().cpu().item())
    image_losses.append(loss_image.detach().cpu().item())
    smooth_losses.append(loss_smooth.detach().cpu().item())

    if epoch % 10 == 0:
        print(
            f'Epoch {epoch}/{finetune_epochs} | loss={loss.item():.6f} | '
            f'image={loss_image.item():.6f} | smooth={loss_smooth.item():.6f} | '
            f'patients={moving_ids.tolist()} -> {fixed_ids.tolist()}'
        )

    if epoch % checkpoint_every == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model3D.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'archive_path': str(archive_path),
            'used_lung_mask': lung_masks is not None,
        }, checkpoint_dir / f'finetune_epoch_{epoch:05d}.pth')

    if epoch % evaluation_every == 0:
        # Recompute after optimizer.step() so the display corresponds to the saved state.
        model3D.eval()
        with torch.no_grad():
            transformed_eval, Vec_eval, _, _, _ = reconstruct_warped(moving_images, fixed_images)
            mse_before = masked_mse(fixed_images, moving_images, overlap_mask).item()
            mse_after = masked_mse(fixed_images, transformed_eval, overlap_mask).item()
            mae_before = (
                (fixed_images - moving_images).abs().mean()
                if overlap_mask is None else
                ((fixed_images - moving_images).abs() * overlap_mask).sum() / overlap_mask.sum().clamp_min(1e-6)
            ).item()
            mae_after = (
                (fixed_images - transformed_eval).abs().mean()
                if overlap_mask is None else
                ((fixed_images - transformed_eval).abs() * overlap_mask).sum() / overlap_mask.sum().clamp_min(1e-6)
            ).item()
        model3D.train()

        print(f'\n--- Epoch {epoch} evaluation ---')
        print(f'MSE before/after: {mse_before:.6f} / {mse_after:.6f}')
        print(f'MAE before/after: {mae_before:.6f} / {mae_after:.6f}')
        print(
            f'Vec min/max/mean|abs|: {Vec_eval.min().item():.3f} / '
            f'{Vec_eval.max().item():.3f} / {Vec_eval.abs().mean().item():.6f}'
        )

        slice_idx = min(64, moving_images.shape[2] - 1)
        plt.figure(figsize=(12, 4))
        for panel, (title, image) in enumerate([
            ('Moving', moving_images[0, 0, slice_idx]),
            ('Fixed', fixed_images[0, 0, slice_idx]),
            ('Warped', transformed_eval[0, 0, slice_idx]),
        ], start=1):
            plt.subplot(1, 3, panel)
            plt.imshow(image.detach().cpu(), cmap='gray')
            plt.title(title)
            plt.axis('off')
        plt.tight_layout()
        plt.show()

        if mse_after >= mse_before:
            print('Warning: this sampled pair did not improve; inspect the displayed image and flow range.')

final_path = checkpoint_dir / 'finetuned_different_patients_final.pth'
torch.save({
    'epoch': finetune_epochs,
    'model_state_dict': model3D.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'archive_path': str(archive_path),
    'used_lung_mask': lung_masks is not None,
}, final_path)
print(f'Fine-tuned model saved: {final_path}')


## Seminar figures

After the 30,000-iteration run, execute the following cells. They load the final checkpoint (or the 30,000 checkpoint as a fallback) without retraining.


In [ ]:
# Seminar visualization: registration before/after using the completed fine-tuning checkpoint.
# Run the setup, wavelet-pipeline, and fine-tuning cells once before this cell.
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

checkpoint_dir = Path('finetune_checkpoints_different_patients')
final_checkpoint = checkpoint_dir / 'finetuned_different_patients_final.pth'
epoch_30000_checkpoint = checkpoint_dir / 'finetune_epoch_30000.pth'
evaluation_checkpoint = final_checkpoint if final_checkpoint.exists() else epoch_30000_checkpoint
if not evaluation_checkpoint.exists():
    raise FileNotFoundError(
        'Final checkpoint was not found. Expected one of:\n'
        f'  {final_checkpoint.resolve()}\n  {epoch_30000_checkpoint.resolve()}'
    )

try:
    state = torch.load(evaluation_checkpoint, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(evaluation_checkpoint, map_location=device)
model3D.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
model3D.eval()

# A fixed pair makes the figure reproducible. Change these IDs only when comparing another case.
moving_patient_id, fixed_patient_id = 0, 1
if len(volumes) < 2:
    raise ValueError('At least two patients are required for visualization.')
if moving_patient_id == fixed_patient_id or max(moving_patient_id, fixed_patient_id) >= len(volumes):
    raise ValueError(f'Invalid patient IDs for {len(volumes)} volumes.')

moving_images = torch.from_numpy(volumes[moving_patient_id:moving_patient_id + 1]).unsqueeze(1).to(device)
fixed_images = torch.from_numpy(volumes[fixed_patient_id:fixed_patient_id + 1]).unsqueeze(1).to(device)
overlap_mask = None
if lung_masks is not None:
    overlap_mask = torch.from_numpy(
        lung_masks[moving_patient_id:moving_patient_id + 1] * lung_masks[fixed_patient_id:fixed_patient_id + 1]
    ).unsqueeze(1).to(device)

with torch.no_grad():
    warped_images, flow, _, _, _ = reconstruct_warped(moving_images, fixed_images)
    mse_before = masked_mse(fixed_images, moving_images, overlap_mask).item()
    mse_after = masked_mse(fixed_images, warped_images, overlap_mask).item()

moving_np = moving_images[0, 0].cpu().numpy()
fixed_np = fixed_images[0, 0].cpu().numpy()
warped_np = warped_images[0, 0].cpu().numpy()
before_np = np.abs(fixed_np - moving_np)
after_np = np.abs(fixed_np - warped_np)
flow_np = flow[0].cpu().numpy()

views = [
    ('Axial', lambda a: a[a.shape[0] // 2]),
    ('Coronal', lambda a: a[:, a.shape[1] // 2, :]),
    ('Sagittal', lambda a: a[:, :, a.shape[2] // 2]),
]
image_vmin, image_vmax = np.percentile(np.concatenate([moving_np.ravel(), fixed_np.ravel(), warped_np.ravel()]), [1, 99])
diff_vmax = max(np.percentile(np.concatenate([before_np.ravel(), after_np.ravel()]), 99), 1e-6)

fig, axes = plt.subplots(3, 5, figsize=(18, 11), constrained_layout=True)
for row, (orientation, slicer) in enumerate(views):
    panels = [
        ('Moving', slicer(moving_np), 'gray', image_vmin, image_vmax),
        ('Fixed', slicer(fixed_np), 'gray', image_vmin, image_vmax),
        ('Warped', slicer(warped_np), 'gray', image_vmin, image_vmax),
        ('|Fixed − moving|', slicer(before_np), 'magma', 0, diff_vmax),
        ('|Fixed − warped|', slicer(after_np), 'magma', 0, diff_vmax),
    ]
    for col, (label, image, cmap, vmin, vmax) in enumerate(panels):
        ax = axes[row, col]
        im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f'{orientation}: {label}')
        ax.axis('off')
        if col == 4:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f'Different-patient registration (patient {moving_patient_id} → {fixed_patient_id})\n'
    f'Masked MSE: {mse_before:.5f} → {mse_after:.5f} '
    f'({(1 - mse_after / mse_before) * 100:.1f}% reduction)',
    fontsize=15,
)
output_figure = Path('seminar_registration_before_after.png')
fig.savefig(output_figure, dpi=200, bbox_inches='tight')
plt.show()
print(f'Loaded checkpoint: {evaluation_checkpoint.resolve()}')
print(f'Masked MSE: before={mse_before:.6f}, after={mse_after:.6f}')
print(f'Figure saved to: {output_figure.resolve()}')

# Flow field on the axial middle slice (arrows are y/x displacement on the downsampled grid).
z = flow_np.shape[1] // 2
step = 8
y, x = np.mgrid[0:flow_np.shape[2]:step, 0:flow_np.shape[3]:step]
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(moving_np[moving_np.shape[0] // 2], cmap='gray', vmin=image_vmin, vmax=image_vmax)
ax.quiver(x * 2, y * 2, flow_np[2, z, ::step, ::step] * 2, flow_np[1, z, ::step, ::step] * 2,
          color='cyan', angles='xy', scale_units='xy', scale=1, width=0.0025)
ax.set_title('Axial displacement field (cyan arrows; ×2 to image resolution)')
ax.axis('off')
flow_figure = Path('seminar_displacement_field.png')
fig.savefig(flow_figure, dpi=200, bbox_inches='tight')
plt.show()
print(f'Flow figure saved to: {flow_figure.resolve()}')


In [ ]:
# Optional learning-curve plot. It is available only if the training cell was run in this kernel.
if 'losses' in globals() and len(losses) > 0:
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(1, len(losses) + 1)
    ax.plot(x, losses, label='Total loss', alpha=0.8)
    ax.plot(x, image_losses, label='Image loss', alpha=0.8)
    ax.plot(x, smooth_losses, label='Smoothness loss', alpha=0.8)
    ax.set(xlabel='Fine-tuning iteration', ylabel='Loss', title='Fine-tuning learning curve')
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig('seminar_learning_curve.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Loss history is not in memory. Re-run training or save losses during training to draw this graph.')
